# Fast-SAM3D · Kaggle 双 T4 常驻 Worker

这份 Notebook 将原来的单次 `infer.py` 改成与 003 TripoSR 相同的常驻 Worker 架构。

- Notebook 内核不改系统默认 Python；Fast-SAM3D 固定运行在独立 Python 3.11 `.venv`。
- GPU0 / GPU1 各启动一个长期存活进程，每张 T4 各自只加载一次 `Inference`。
- Worker 通过 `POST /task/claim` 原子领取 `fast-sam3d` 任务，心跳续租。
- 每个任务从 Hub 下载 image + mask，执行 acceleration 路径，导出 GLB。
- GLB 经 AES-GCM 加密后复用 Hub 的 `/upload/artifact` 上传。
- 继续复用原 007 已绑定的 checkpoint、gsplat wheel、pytorch3d wheel Kaggle Input。

Kaggle 要求：**GPU T4 x2 + Internet**。还需要一个 Kaggle Secret：`KAGGLE_HUB_TOKEN`。


## Cell 1 · 安装 Python 3.11 Fast-SAM3D Runtime


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ["UV_LINK_MODE"] = "copy"
os.environ["IPYTHONDIR"] = "/kaggle/working/.ipython"
ROOT = Path("/kaggle/working/Fast-SAM3D")
UV = "/usr/local/bin/uv"
PY311 = Path("/usr/bin/python3.11")
PY = ROOT / ".venv/bin/python"
FASTSAM_REPO = "https://github.com/wlfeng0509/Fast-SAM3D.git"
FASTSAM_COMMIT = "c76188fda755b761fe9255eabe87aaad39df829a"
UTILS3D_REPO = "https://github.com/EasternJournalist/utils3d.git"
UTILS3D_COMMIT = "3913c65d81e05e47b9f367250cf8c0f7462a0900"
MOGE_REPO = "https://github.com/microsoft/MoGe.git"
MOGE_COMMIT = "a8c37341bc0325ca99b9d57981cc3bb2bd3e255b"
KAGGLEHUB_VERSION = "1.0.2"
HF_FASTSAM_REPO = "facebook/sam-3d-objects"
GSPLAT_DATASET = "lhuyton/gsplat-wheel"
GSPLAT_FILENAME = "gsplat-1.5.3-cp311-cp311-linux_x86_64.whl"
PYTORCH3D_DATASET = "lhuyton/pytorch-wheel"
PYTORCH3D_FILENAME = "pytorch3d-0.7.9-cp311-cp311-linux_x86_64.whl"

DOWNLOAD_ROOT = Path("/kaggle/working/.fastsam3d-downloads")
CHECKPOINT_DIR = DOWNLOAD_ROOT / "sam-3d-objects"
WHEEL_DIR = DOWNLOAD_ROOT / "wheels"

def run(cmd, cwd=None, env=None):
    cmd = list(map(str, cmd))
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None, env=env)

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if not HF_TOKEN: raise RuntimeError("Kaggle Secret HF_TOKEN 未配置")

run([os.sys.executable, "-m", "pip", "install", "-q", f"kagglehub=={KAGGLEHUB_VERSION}", "huggingface_hub"])
import kagglehub
from huggingface_hub import snapshot_download

DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
WHEEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_download = Path(snapshot_download(repo_id=HF_FASTSAM_REPO, local_dir=str(CHECKPOINT_DIR), token=HF_TOKEN))
checkpoint_source = checkpoint_download / "checkpoints"
if not checkpoint_source.is_dir(): raise RuntimeError(f"Fast-SAM3D checkpoint 目录不存在: {checkpoint_source}")

GSPLAT_WHEEL = Path(kagglehub.dataset_download(GSPLAT_DATASET, path=GSPLAT_FILENAME, output_dir=str(WHEEL_DIR / "gsplat")))
PYTORCH3D_WHEEL = Path(kagglehub.dataset_download(PYTORCH3D_DATASET, path=PYTORCH3D_FILENAME, output_dir=str(WHEEL_DIR / "pytorch3d")))

for wheel in (GSPLAT_WHEEL, PYTORCH3D_WHEEL):
    if not wheel.is_file(): raise RuntimeError(f"wheel 下载失败: {wheel}")

print("checkpoint:", checkpoint_source)
print("gsplat:", GSPLAT_WHEEL)
print("pytorch3d:", PYTORCH3D_WHEEL)

if not PY311.exists():
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", "python3.11", "python3.11-dev", "python3.11-venv"])

if ROOT.exists(): shutil.rmtree(ROOT)
run(["git", "clone", FASTSAM_REPO, ROOT])
run(["git", "checkout", FASTSAM_COMMIT], cwd=ROOT)
run([UV, "venv", "--no-managed-python", "--python", PY311, ROOT / ".venv"])

pip_env = os.environ.copy()
pip_env["PIP_EXTRA_INDEX_URL"] = "https://pypi.ngc.nvidia.com https://download.pytorch.org/whl/cu121"
pip_env["PIP_FIND_LINKS"] = "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html"
pip_env["MAX_JOBS"] = "4"

run([UV, "pip", "install", "--python", PY, "setuptools==69.5.1", "wheel", "hatchling", "hatch-requirements-txt"], env=pip_env)
run([UV, "pip", "install", "--python", PY, "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121"], env=pip_env)
run([UV, "pip", "install", "--python", PY, "-e", f"{ROOT}[dev]"], env=pip_env)

utils_root = Path("/kaggle/working/utils3d")
if utils_root.exists(): shutil.rmtree(utils_root)
run(["git", "clone", UTILS3D_REPO, utils_root])
run(["git", "checkout", UTILS3D_COMMIT], cwd=utils_root)
pt_alias = utils_root / "utils3d/pt"
if not pt_alias.exists(): os.symlink("torch", pt_alias, target_is_directory=True)
run([UV, "pip", "install", "--python", PY, str(utils_root)], env=pip_env)

moge_root = Path("/kaggle/working/MoGe")
if moge_root.exists(): shutil.rmtree(moge_root)
run(["git", "clone", MOGE_REPO, moge_root])
run(["git", "checkout", MOGE_COMMIT], cwd=moge_root)

requirements = moge_root / "requirements.txt"
requirements.write_text("\n".join("# " + line if line.strip().startswith("git+https://github.com/EasternJournalist/utils3d.git") else line for line in requirements.read_text().splitlines()) + "\n")

pyproject = moge_root / "pyproject.toml"
pyproject.write_text(pyproject.read_text().replace('"utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900"', '"utils3d"'))

run([UV, "pip", "install", "--python", PY, str(moge_root)], env=pip_env)
run([UV, "pip", "install", "--python", PY, GSPLAT_WHEEL], env=pip_env)
run([UV, "pip", "install", "--python", PY, PYTORCH3D_WHEEL], env=pip_env)
run([UV, "pip", "install", "--python", PY, "kaolin==0.17.0", "--find-links", "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html"], env=pip_env)
run([UV, "pip", "install", "--python", PY, "flash_attn==2.8.3", "--no-build-isolation"], env=pip_env)
run([UV, "pip", "install", "--python", PY, "warp-lang", "spconv-cu121", "huggingface_hub", "cryptography", "requests"], env=pip_env)

moge_checkpoint = ROOT / "checkpoints/moge-vitl"
moge_checkpoint.mkdir(parents=True, exist_ok=True)
run([PY, "-c", f"from huggingface_hub import snapshot_download; snapshot_download(repo_id='Ruicheng/moge-vitl', local_dir={str(moge_checkpoint)!r}, token={HF_TOKEN!r})"])

target = ROOT / "notebook/checkpoints/hf"
target.mkdir(parents=True, exist_ok=True)

for child in target.iterdir():
    if child.is_symlink() or child.is_file(): child.unlink()
    elif child.is_dir(): shutil.rmtree(child)

for src in checkpoint_source.iterdir(): os.symlink(src, target / src.name, target_is_directory=src.is_dir())

print("Using pinned upstream inference_pipeline.py")

for path in ROOT.rglob("*.py"):
    text = path.read_text(encoding="utf-8", errors="ignore")
    replaced = text.replace("/data3/wmq/Fast-sam3d-objects/checkpoints/torch-cache", "/kaggle/working/torch-cache")
    if replaced != text: path.write_text(replaced, encoding="utf-8")

dino_cache = Path("/kaggle/working/torch-cache/hub/facebookresearch_dinov2_main")
dino_cache.parent.mkdir(parents=True, exist_ok=True)
if dino_cache.exists(): shutil.rmtree(dino_cache)
run(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/dinov2.git", dino_cache])

dino = ROOT / "notebook/facebookresearch/dinov2"
dino.parent.mkdir(parents=True, exist_ok=True)
if dino.exists() or dino.is_symlink():
    if dino.is_symlink() or dino.is_file(): dino.unlink()
    else: shutil.rmtree(dino)

os.symlink(dino_cache, dino, target_is_directory=True)

run([PY, "-c", "import torch, sam3d_objects, pytorch3d, gsplat, kaolin, moge; print('runtime OK', torch.__version__, torch.version.cuda)"])
print("✅ Fast-SAM3D Python 3.11 runtime ready:", PY)


>>> /usr/bin/python3 -m pip install -q kagglehub==1.0.2 huggingface_hub


Fetching 31 files:   0%|          | 0/31 [00:00<?, ?it/s]

checkpoint: /kaggle/working/.fastsam3d-downloads/sam-3d-objects/checkpoints
gsplat: /kaggle/input/datasets/lhuyton/gsplat-wheel/gsplat-1.5.3-cp311-cp311-linux_x86_64.whl
pytorch3d: /kaggle/input/datasets/lhuyton/pytorch-wheel/pytorch3d-0.7.9-cp311-cp311-linux_x86_64.whl

>>> apt-get update
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [112 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 https://ppa.launchpadcon

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libpython3.11 libpython3.11-dev libpython3.11-minimal libpython3.11-stdlib
  python3.11-minimal
Suggested packages:
  binfmt-support
The following NEW packages will be installed:
  libpython3.11 libpython3.11-dev libpython3.11-minimal libpython3.11-stdlib
  python3.11 python3.11-dev python3.11-minimal python3.11-venv
0 upgraded, 8 newly installed, 0 to remove and 197 not upgraded.
Need to get 16.5 MB of archives.
After this operation, 58.4 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.11-minimal amd64 3.11.15-1+jammy1 [887 kB]
Get:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 python3.11-minimal amd64 3.11.15-1+jammy1 [2,353 kB]
Get:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.11-stdlib amd64 3.11.15-1+jammy1 [1,927 kB]
Get:4

...

Note: switching to 'c76188fda755b761fe9255eabe87aaad39df829a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at c76188f Update README.md
Using CPython 3.11.15 interpreter at: /usr/bin/python3.11
Creating virtual environment at: Fast-SAM3D/.venv
Activate with: source Fast-SAM3D/.venv/bin/activate
Using Python 3.11.15 environment at: Fast-SAM3D/.venv



>>> /usr/local/bin/uv pip install --python /kaggle/working/Fast-SAM3D/.venv/bin/python setuptools==69.5.1 wheel hatchling hatch-requirements-txt


Resolved 9 packages in 407ms
Prepared 9 packages in 179ms
Installed 9 packages in 52ms
 + hatch-requirements-txt==0.4.1
 + hatchling==1.32.0
 + packaging==26.3
 + pathspec==1.1.1
 + pluggy==1.6.0
 + setuptools==69.5.1
 + tomlkit==0.15.1
 + trove-classifiers==2026.6.1.19
 + wheel==0.48.0
Using Python 3.11.15 environment at: Fast-SAM3D/.venv



>>> /usr/local/bin/uv pip install --python /kaggle/working/Fast-SAM3D/.venv/bin/python torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121


Resolved 26 packages in 1.14s


## Cell 2 · 写入常驻双 GPU Worker

Worker 源码与仓库 `notebooks/fast_sam3d_worker.py` 保持同步。


In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/Fast-SAM3D")
WORKER = ROOT / "kaggle_worker.py"

WORKER_SOURCE = r'''from __future__ import annotations

import gc
import hashlib
import io
import multiprocessing as mp
import os
import signal
import socket
import sys
import tempfile
import threading
import time
import traceback
import uuid
from argparse import Namespace
from pathlib import Path
from typing import Any
from urllib.parse import urljoin

import numpy as np
import requests
import torch
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from PIL import Image

BASE_URL = os.environ["BASE_URL"].rstrip("/") + "/"
TOKEN = os.environ["KAGGLE_HUB_TOKEN"]
ROOT = Path(os.getenv("FAST_SAM3D_ROOT", "/kaggle/working/Fast-SAM3D"))
NOTEBOOK_DIR = ROOT / "notebook"
CHECKPOINT_DIR = Path(os.getenv("FAST_SAM3D_CHECKPOINT_DIR", str(NOTEBOOK_DIR / "checkpoints/hf")))
TORCH_CACHE = Path(os.getenv("FAST_SAM3D_TORCH_CACHE", "/kaggle/working/torch-cache"))
MODEL = "fast-sam3d"
POLL_TIMEOUT = 35
REQUEST_TIMEOUT = 180
HEARTBEAT_SECONDS = 10
IDLE_DIAGNOSTIC_SECONDS = 60


def encrypt_blob(data: bytes) -> bytes:
    key = hashlib.sha256(TOKEN.encode()).digest()
    nonce = os.urandom(12)
    return nonce + AESGCM(key).encrypt(nonce, data, None)


def api_url(path: str) -> str:
    return urljoin(BASE_URL, path.lstrip("/"))


def auth_headers() -> dict[str, str]:
    return {
        "Authorization": f"Bearer {TOKEN}",
        "Cache-Control": "no-cache, no-store",
        "Pragma": "no-cache",
    }


def checked_response(response: requests.Response, label: str) -> requests.Response:
    if response.status_code >= 400:
        body = response.text[:1000].replace("\n", " ")
        raise RuntimeError(f"{label}: HTTP {response.status_code} | {body}")
    return response


def server_snapshot(session: requests.Session) -> dict[str, Any]:
    response = session.get(api_url("/api/status"), params={"_ts": time.time_ns()}, timeout=15)
    checked_response(response, "GET /api/status")
    return response.json()


def preflight_hub() -> str:
    session = requests.Session()
    session.headers.update(auth_headers())
    response = session.get(api_url("/api/models"), params={"_ts": time.time_ns()}, timeout=20)
    checked_response(response, "GET /api/models")
    model_ids = {item.get("id") for item in response.json() if isinstance(item, dict)}
    if MODEL not in model_ids:
        raise RuntimeError(f"Hub does not advertise {MODEL}. Update/restart Hub first.")
    checked_response(
        session.get(api_url("/api/failed"), params={"_ts": time.time_ns()}, timeout=20),
        "GET /api/failed (auth check)",
    )
    snapshot = server_snapshot(session)
    if snapshot.get("storage") != "sqlite":
        raise RuntimeError("Connected Hub is an old in-memory build. Update/restart Hub first.")
    instance_id = str(snapshot.get("hub_instance_id") or "")
    if not instance_id:
        raise RuntimeError("Hub did not return hub_instance_id; protocol versions differ")
    queued = int(snapshot.get("queued_by_model", {}).get(MODEL, 0) or 0)
    inflight = int(snapshot.get("inflight_by_model", {}).get(MODEL, 0) or 0)
    print(
        f"[preflight] Hub OK | instance={instance_id[:12]} | storage=sqlite | "
        f"{MODEL} queued={queued} inflight={inflight}",
        flush=True,
    )
    return instance_id


def validate_runtime() -> None:
    pipeline = CHECKPOINT_DIR / "pipeline.yaml"
    if not ROOT.is_dir():
        raise RuntimeError(f"Fast-SAM3D root not found: {ROOT}")
    if not pipeline.is_file():
        raise RuntimeError(f"Fast-SAM3D checkpoint config not found: {pipeline}")
    if not (NOTEBOOK_DIR / "inference.py").is_file():
        raise RuntimeError(f"Fast-SAM3D inference.py not found under {NOTEBOOK_DIR}")
    TORCH_CACHE.mkdir(parents=True, exist_ok=True)


def _rewrite_paths(value: Any) -> Any:
    if isinstance(value, str):
        replacements = {
            "/data3/wmq/Fast-sam3d-objects/checkpoints/torch-cache": str(TORCH_CACHE),
            "/data3/wmq/Fast-sam3d-objects/checkpoints/": str(NOTEBOOK_DIR / "checkpoints") + "/",
        }
        for old, new in replacements.items():
            value = value.replace(old, new)
        return value
    if isinstance(value, dict):
        return {key: _rewrite_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_rewrite_paths(item) for item in value]
    return value


def inference_args(enable_acceleration: bool = True) -> Namespace:
    return Namespace(
        ss_faster_stride=3,
        ss_warmup=2,
        ss_order=1,
        ss_momentum_beta=0.5,
        slat_thresh=0.5,
        slat_warmup=2,
        slat_token_ratio=0.15,
        mesh_spectral_threshold_low=0.5,
        mesh_spectral_threshold_high=0.7,
        enable_ss_faster=enable_acceleration,
        enable_slat_token=enable_acceleration,
        enable_mesh_aggregation=enable_acceleration,
        enable_acceleration=enable_acceleration,
        enable_taylor=False,
        enable_easy=False,
    )


def build_inference(enable_acceleration: bool = True):
    os.environ.setdefault("CONDA_PREFIX", "/opt/conda")
    os.environ["TORCH_HOME"] = str(TORCH_CACHE)
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    sys.path.insert(0, str(NOTEBOOK_DIR))
    sys.path.insert(0, str(ROOT))
    os.chdir(NOTEBOOK_DIR)

    from omegaconf import OmegaConf
    from inference import Inference

    config = OmegaConf.load(CHECKPOINT_DIR / "pipeline.yaml")
    plain = OmegaConf.to_container(config, resolve=False)
    config = OmegaConf.create(_rewrite_paths(plain))
    config.workspace_dir = str(CHECKPOINT_DIR)
    if enable_acceleration:
        config["ss_generator_config_path"] = "ss_generator_faster.yaml"
        config["slat_generator_config_path"] = "slat_generator_faster.yaml"

    args = inference_args(enable_acceleration)
    inference = Inference(config, compile=False, args=args)
    if hasattr(inference, "get_params"):
        inference.get_params(args)
    return inference


def prepare_inputs(image_raw: bytes, mask_raw: bytes, directory: Path):
    from fft.fft2d import calculate_hfer_robust

    image = Image.open(io.BytesIO(image_raw)).convert("RGB")
    mask_image = Image.open(io.BytesIO(mask_raw)).convert("L")
    if mask_image.size != image.size:
        raise ValueError(f"Mask size {mask_image.size} must match image size {image.size}")
    mask_path = directory / "mask.png"
    mask_image.save(mask_path)
    image_array = np.asarray(image, dtype=np.uint8)
    mask_array = np.asarray(mask_image, dtype=np.uint8) > 0
    if not mask_array.any():
        raise ValueError("Mask is empty")
    hfer = calculate_hfer_robust(str(mask_path))
    return image_array, mask_array, hfer


def export_glb(glb: Any, path: Path) -> bytes:
    glb.export(str(path))
    data = path.read_bytes()
    if len(data) < 12 or data[:4] != b"glTF":
        raise RuntimeError("Fast-SAM3D export did not produce a valid GLB")
    return data


def heartbeat_loop(
    worker_id: str,
    gpu: int,
    stop: threading.Event,
    active_task: dict[str, int | None],
) -> None:
    session = requests.Session()
    session.headers.update(auth_headers())
    while not stop.wait(HEARTBEAT_SECONDS):
        try:
            session.post(
                api_url("/worker/heartbeat"),
                json={
                    "worker_id": worker_id,
                    "local_queue": 0,
                    "upload_queue": 0,
                    "active_task_id": active_task["id"],
                    "meta": {"gpu_index": gpu, "persistent": True},
                },
                timeout=15,
            ).raise_for_status()
        except Exception as exc:
            print(f"[GPU{gpu}] heartbeat: {type(exc).__name__}: {exc}", flush=True)


def report_failure(session: requests.Session, task_id: int, exc: BaseException, gpu: int) -> None:
    message = f"{type(exc).__name__}: {exc}"
    print(f"[GPU{gpu}] FAIL #{task_id}: {message}", flush=True)
    try:
        session.post(
            api_url("/task/fail"),
            json={"id": task_id, "error": message[:1900], "requeue": True},
            timeout=20,
        ).raise_for_status()
    except Exception as report_exc:
        print(f"[GPU{gpu}] fail-report error: {report_exc}", flush=True)


def gpu_worker(gpu: int, run_id: str, hub_instance_id: str) -> None:
    torch.cuda.set_device(gpu)
    gpu_name = torch.cuda.get_device_name(gpu)
    worker_id = f"fast-sam3d-{run_id}-g{gpu}"

    print(f"[GPU{gpu}] loading Fast-SAM3D on {gpu_name} ...", flush=True)
    started = time.perf_counter()
    inference = build_inference(enable_acceleration=True)
    print(f"[GPU{gpu}] READY in {time.perf_counter()-started:.2f}s", flush=True)

    session = requests.Session()
    session.headers.update(auth_headers())
    register_response = session.post(
        api_url("/worker/register"),
        json={
            "worker_id": worker_id,
            "model": MODEL,
            "gpus": [gpu_name],
            "runtime": "fast-sam3d-persistent-py311",
            "concurrency": 1,
            "meta": {
                "gpu_index": gpu,
                "torch": torch.__version__,
                "torch_cuda": torch.version.cuda,
                "persistent": True,
                "checkpoint_dir": str(CHECKPOINT_DIR),
            },
        },
        timeout=30,
    )
    checked_response(register_response, "POST /worker/register")
    print(f"[GPU{gpu}] registered as {worker_id}", flush=True)

    stop = threading.Event()
    active_task: dict[str, int | None] = {"id": None}
    heartbeat = threading.Thread(
        target=heartbeat_loop,
        args=(worker_id, gpu, stop, active_task),
        daemon=True,
    )
    heartbeat.start()

    def shutdown(*_args):
        stop.set()
        raise KeyboardInterrupt

    signal.signal(signal.SIGTERM, shutdown)
    last_idle_diagnostic = 0.0

    try:
        while True:
            task: dict[str, Any] | None = None
            try:
                response = session.post(
                    api_url("/task/claim"),
                    json={"model": MODEL, "worker_id": worker_id, "wait_seconds": 25},
                    timeout=POLL_TIMEOUT,
                )
                response_instance = response.headers.get("X-Hub-Instance", "")
                if response_instance and response_instance != hub_instance_id:
                    raise RuntimeError(
                        f"Hub instance changed: expected={hub_instance_id[:12]} got={response_instance[:12]}"
                    )
                if response.status_code == 204:
                    now = time.monotonic()
                    if now - last_idle_diagnostic >= IDLE_DIAGNOSTIC_SECONDS:
                        last_idle_diagnostic = now
                        snapshot = server_snapshot(session)
                        queued = int(snapshot.get("queued_by_model", {}).get(MODEL, 0) or 0)
                        inflight = int(snapshot.get("inflight_by_model", {}).get(MODEL, 0) or 0)
                        print(
                            f"[GPU{gpu}] idle | Hub {MODEL} queued={queued} inflight={inflight} "
                            f"instance={str(snapshot.get('hub_instance_id', ''))[:12]}",
                            flush=True,
                        )
                    continue
                checked_response(response, "POST /task/claim")
                task = response.json()
                task_id = int(task["id"])
                active_task["id"] = task_id
                seed = int(task.get("seed", 42))
                print(
                    f"[GPU{gpu}] ↓ #{task_id} {task.get('source_label', 'input')} seed={seed}",
                    flush=True,
                )

                t0 = time.perf_counter()
                image_response = session.get(api_url(task["input_url"]), timeout=60)
                image_response.raise_for_status()
                mask_response = session.get(api_url(task["mask_url"]), timeout=60)
                mask_response.raise_for_status()
                t_download = time.perf_counter()

                with tempfile.TemporaryDirectory(prefix=f"fastsam3d-{task_id}-") as temp_dir:
                    temp = Path(temp_dir)
                    image, mask, hfer = prepare_inputs(image_response.content, mask_response.content, temp)
                    t_pre = time.perf_counter()

                    if hasattr(inference, "get_hfer"):
                        inference.get_hfer(hfer)
                    with torch.inference_mode():
                        output = inference(image, mask, seed=seed)
                    t_model = time.perf_counter()

                    artifact = export_glb(output["glb"], temp / "result.glb")
                    encrypted = encrypt_blob(artifact)
                    t_export = time.perf_counter()

                elapsed = t_export - t0
                upload = session.post(
                    api_url("/upload/artifact"),
                    data={
                        "id": str(task_id),
                        "model": MODEL,
                        "worker_id": worker_id,
                        "gpu": str(gpu),
                        "seconds": f"{elapsed:.3f}",
                        "output_format": "glb",
                    },
                    files={
                        "file": (
                            f"{task_id}.glb.bin",
                            encrypted,
                            "application/octet-stream",
                        )
                    },
                    timeout=REQUEST_TIMEOUT,
                )
                checked_response(upload, "POST /upload/artifact")
                t_upload = time.perf_counter()
                active_task["id"] = None

                print(
                    f"[GPU{gpu}] ✓ #{task_id} total={elapsed:.2f}s "
                    f"download={t_download-t0:.2f}s preprocess={t_pre-t_download:.2f}s "
                    f"model={t_model-t_pre:.2f}s export={t_export-t_model:.2f}s "
                    f"upload={t_upload-t_export:.2f}s",
                    flush=True,
                )

                del image, mask, hfer, output, artifact, encrypted
                gc.collect()
                torch.cuda.empty_cache()

            except KeyboardInterrupt:
                raise
            except Exception as exc:
                if task is not None and "id" in task:
                    report_failure(session, int(task["id"]), exc, gpu)
                    active_task["id"] = None
                else:
                    print(f"[GPU{gpu}] poll error: {type(exc).__name__}: {exc}", flush=True)
                    time.sleep(2)
                traceback.print_exc()
    except KeyboardInterrupt:
        pass
    finally:
        stop.set()
        print(f"[GPU{gpu}] stopped", flush=True)


def main() -> None:
    validate_runtime()
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")
    gpu_count = torch.cuda.device_count()
    if gpu_count < 1:
        raise RuntimeError("No CUDA GPU found")
    wanted = int(os.getenv("FAST_SAM3D_GPU_COUNT", str(gpu_count)))
    gpu_count = min(gpu_count, max(1, wanted))
    run_id = f"{socket.gethostname()[:8]}-{uuid.uuid4().hex[:6]}"

    print(f"Fast-SAM3D persistent worker | GPUs={gpu_count} | base={BASE_URL}", flush=True)
    hub_instance_id = preflight_hub()

    ctx = mp.get_context("spawn")
    processes = [
        ctx.Process(target=gpu_worker, args=(gpu, run_id, hub_instance_id), name=f"fast-sam3d-gpu{gpu}")
        for gpu in range(gpu_count)
    ]
    for process in processes:
        process.start()

    try:
        for process in processes:
            process.join()
    except KeyboardInterrupt:
        print("Stopping workers ...", flush=True)
        for process in processes:
            if process.is_alive():
                process.terminate()
        for process in processes:
            process.join(timeout=10)


if __name__ == "__main__":
    mp.freeze_support()
    main()
'''
WORKER.write_text(WORKER_SOURCE, encoding="utf-8")
print(f"✅ wrote {WORKER} ({len(WORKER_SOURCE.splitlines())} lines)")


## Cell 3 · 启动双 T4 常驻 Worker

启动前 Worker 会校验 Hub 是否已经公开 `fast-sam3d` 模型和 SQLite 共享队列。


In [ ]:
import os
import subprocess
from pathlib import Path
ROOT = Path("/kaggle/working/Fast-SAM3D")
PY = ROOT / ".venv/bin/python"
WORKER = ROOT / "kaggle_worker.py"
LOG = Path("/kaggle/working/fast-sam3d-worker.log")
PID_FILE = Path("/kaggle/working/fast-sam3d-worker.pid")
BASE_URL = os.getenv("KAGGLE_HUB_BASE_URL", "https://ranran-sana.202820.xyz").rstrip("/")
TOKEN = os.getenv("KAGGLE_HUB_TOKEN", "")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("KAGGLE_HUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("请在 Kaggle Secrets 创建 KAGGLE_HUB_TOKEN，或先设置 os.environ['KAGGLE_HUB_TOKEN']") from exc
if PID_FILE.exists():
    try:
        old_pid = int(PID_FILE.read_text().strip()); os.kill(old_pid, 0)
        raise RuntimeError(f"Worker 已在运行 PID={old_pid}；先执行停止 Cell")
    except ProcessLookupError: PID_FILE.unlink(missing_ok=True)
env = os.environ.copy()
env.update({"BASE_URL": BASE_URL, "KAGGLE_HUB_TOKEN": TOKEN, "FAST_SAM3D_ROOT": str(ROOT), "FAST_SAM3D_GPU_COUNT": "2", "PYTHONUNBUFFERED": "1", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
log_handle = LOG.open("ab", buffering=0)
process = subprocess.Popen([str(PY), str(WORKER)], cwd=str(ROOT), env=env, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
PID_FILE.write_text(str(process.pid))
print("✅ Fast-SAM3D worker started | PID", process.pid)
print("log:", LOG)


## Cell 4 · 查看 Worker / Hub / 队列状态


In [ ]:
import json
import os
import time
from pathlib import Path
from urllib.request import Request, urlopen
PID_FILE = Path("/kaggle/working/fast-sam3d-worker.pid")
LOG = Path("/kaggle/working/fast-sam3d-worker.log")
BASE_URL = os.getenv("KAGGLE_HUB_BASE_URL", "https://ranran-sana.202820.xyz").rstrip("/")
TOKEN = os.getenv("KAGGLE_HUB_TOKEN", "")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("KAGGLE_HUB_TOKEN")
    except Exception: TOKEN = ""
if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try: os.kill(pid, 0); print("Worker: RUNNING | PID", pid)
    except ProcessLookupError: print("Worker: EXITED | PID", pid)
else: print("Worker: NOT STARTED")
if TOKEN:
    req = Request(f"{BASE_URL}/api/status?_ts={time.time_ns()}", headers={"Authorization": f"Bearer {TOKEN}", "Cache-Control": "no-cache, no-store"})
    try:
        with urlopen(req, timeout=20) as response: snapshot = json.load(response)
        print("Hub:", snapshot.get("hub_instance_id"), "storage=", snapshot.get("storage"))
        print("fast-sam3d queued=", snapshot.get("queued_by_model", {}).get("fast-sam3d", 0), "inflight=", snapshot.get("inflight_by_model", {}).get("fast-sam3d", 0))
        workers = [w for w in snapshot.get("workers", []) if w.get("model") == "fast-sam3d"]
        print("Fast-SAM3D workers:", len(workers))
        for worker in workers: print(" -", worker.get("worker_id"), "online=", worker.get("online"), "active=", worker.get("active_task_id"))
    except Exception as exc: print("Hub status error:", type(exc).__name__, exc)
else: print("Hub status skipped: KAGGLE_HUB_TOKEN unavailable")
print("\n=== LOG TAIL ===")
print("\n".join(LOG.read_text(errors="replace").splitlines()[-100:]) if LOG.exists() else "no log yet")


## Cell 5 · 停止 Worker


In [ ]:
# import os
# import signal
# import time
# from pathlib import Path
# PID_FILE = Path("/kaggle/working/fast-sam3d-worker.pid")
# if not PID_FILE.exists():
#     print("没有运行中的 Worker")
# else:
#     pid = int(PID_FILE.read_text().strip())
#     try:
#         os.killpg(pid, signal.SIGTERM)
#         print("Stopping process group:", pid)
#         time.sleep(2)
#     except ProcessLookupError:
#         pass
#     PID_FILE.unlink(missing_ok=True)
#     print("✅ stopped")


## 运行方式

1. 本地更新并重启 Hub，确保 `/api/models` 已包含 `fast-sam3d`。
2. Kaggle Notebook 绑定原 007 使用的三个 Input：SAM3D checkpoint、gsplat cp311 wheel、pytorch3d cp311 wheel。
3. 在 Kaggle Secrets 创建 `KAGGLE_HUB_TOKEN`；Hub 地址可用 `KAGGLE_HUB_BASE_URL` 覆盖。
4. 依次执行 Cell 1 → 2 → 3；Cell 4 应看到两个 `fast-sam3d-*` Worker 在线。
5. Hub 端提交一张 RGB image 和同尺寸 mask。Worker 会常驻处理并把 GLB 返回 3D 资产库。

注意：Fast-SAM3D **不是** TripoSR 的“无 mask 单图”接口。上游推理至少需要 RGB + mask；mask 非空且尺寸必须与 RGB 一致。
